In [ ]:
import kagglehub
import os
import pandas as pd
import re
import unicodedata
from transformers import AutoTokenizer
from datasets import Dataset
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download(
    "fredericods/ptbr-sentiment-analysis-datasets"
)

import os
os.listdir(path)

df = pd.read_csv(os.path.join(path, "b2w.csv"))

df.info()

df.describe()

df.head()

df = df.drop(columns=["review_text_processed", "review_text_tokenized"])

df.columns

# removendso null e duplicadas
print(df.isnull().sum())

df = df.dropna(subset=["polarity"])
print("Duplicados:", df.duplicated(subset=["review_text"]).sum())

df = df.drop_duplicates(subset=["review_text"])

print(df["polarity"].value_counts())

df["label"] = df["polarity"].astype(int)

print(df["label"].value_counts())
df[["polarity", "label"]].head()

#preprocessamento
def preprocess(text):
    text = str(text).lower()

    # remove acentos
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ASCII", "ignore").decode("utf-8")

    # remove links
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove quebras de linha
    text = text.replace("\n", " ")

    # remove espaços repetidos
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["review_text_processed"] = df["review_text"].apply(preprocess)

df[["review_text","review_text_processed", "polarity", "label"]].head(40)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1765,
    random_state=42,
    stratify=train_df["label"]
)

print(f"Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")

# tokenização modelo 1
model_name_1 = "neuralmind/bert-base-portuguese-cased"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)

def tokenize_function_1(examples):
    return tokenizer_1(
        examples["review_text_processed"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset_1 = Dataset.from_pandas(train_df)
val_dataset_1 = Dataset.from_pandas(val_df)
test_dataset_1 = Dataset.from_pandas(test_df)

tokenized_train_1 = train_dataset_1.map(tokenize_function_1, batched=True)
tokenized_val_1 = val_dataset_1.map(tokenize_function_1, batched=True)
tokenized_test_1 = test_dataset_1.map(tokenize_function_1, batched=True)

if "__index_level_0__" in tokenized_train_1.column_names:
    tokenized_train_1 = tokenized_train_1.remove_columns(["__index_level_0__"])
    tokenized_val_1 = tokenized_val_1.remove_columns(["__index_level_0__"])
    tokenized_test_1 = tokenized_test_1.remove_columns(["__index_level_0__"])

tokenized_train_1.to_pandas().head()
tokenized_val_1.to_pandas().head()
tokenized_test_1.to_pandas().head()

# tokenização modelo 2
model_name_2 = "bert-base-multilingual-cased"

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)

def tokenize_function_2(examples):
    return tokenizer_2(
        examples["review_text_processed"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset_2 = Dataset.from_pandas(train_df)
val_dataset_2 = Dataset.from_pandas(val_df)
test_dataset_2 = Dataset.from_pandas(test_df)

tokenized_train_2 = train_dataset_2.map(tokenize_function_2, batched=True)
tokenized_val_2 = val_dataset_2.map(tokenize_function_2, batched=True)
tokenized_test_2 = test_dataset_2.map(tokenize_function_2, batched=True)

if "__index_level_0__" in tokenized_train_2.column_names:
    tokenized_train_2 = tokenized_train_2.remove_columns(["__index_level_0__"])
    tokenized_val_2 = tokenized_val_2.remove_columns(["__index_level_0__"])
    tokenized_test_2 = tokenized_test_2.remove_columns(["__index_level_0__"])

tokenized_train_2.to_pandas().head()
tokenized_val_2.to_pandas().head()
tokenized_test_2.to_pandas().head()

## Experimentos de Fine-Tuning
Tive problemas com o Tempo e o limite de execução utilizando a GPU e versão free do Colab então eu fiz que
cada resultado de treinamento é salvo em
um CSV assim que fica pronto. Se o Colab desconectar ou a cota de GPU acabar no meio do processo,  não perde o que já rodou ao reexecutar o notebook, os experimentos já concluídos são carregados em vez de retreinados por conta .
Experimento 4 mas  em vez de treinar o modelo completo 3 vezes
(2, 3 e 5 épocas) usando as ~75 mil amostras de treino inteiras a busca de número de épocas usa uma subamostra estratificada por rótulo do conjunto de treino.

## 1. Imports e configuração

In [ ]:
import gc
import inspect
import os
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo em uso: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("ATENÇÃO: nenhuma GPU detectada. Ative Ambiente de execução > "
          "Alterar tipo de ambiente de execução > Acelerador de hardware > GPU (T4).")

_TA_PARAMS = inspect.signature(TrainingArguments.__init__).parameters
EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"

## 2. Persistência dos resultados (cache resumível)

**Recomendado: monte o Google Drive** para que o arquivo de resultados
sobreviva a quedas de sessão e você possa continuar de onde parou em outro
dia (a cota de GPU do Colab gratuito é renovada periodicamente).

Se preferir não usar o Drive, deixe `USE_DRIVE = False` — o cache ainda
protege contra falhas *dentro* da mesma sessão, mas será perdido se o
ambiente de execução for reiniciado.

In [ ]:
USE_DRIVE = True  # recomendado: mude para False se não quiser usar o Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_PATH = "/content/drive/MyDrive/tp2_resultados_experimentos.csv"
else:
    RESULTS_PATH = "resultados_experimentos.csv"

print(f"Resultados serão salvos/lidos em: {RESULTS_PATH}")


def load_cached_results():
    """Carrega os resultados já salvos (lista de dicts). Vazio se não existir."""
    if os.path.exists(RESULTS_PATH):
        df = pd.read_csv(RESULTS_PATH)
        print(f"Cache encontrado: {len(df)} experimento(s) já salvo(s).")
        return df.to_dict("records")
    print("Nenhum cache encontrado ainda — começando do zero.")
    return []


def save_cached_results(results):
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)


def run_or_load(cache, run_name, **kwargs):
    """Retorna o resultado do cache se `run_name` já foi treinado; caso
    contrário, treina agora com `run_experiment` e salva no cache.
    """
    for r in cache:
        if r["run_name"] == run_name:
            print(f"[cache] '{run_name}' já treinado — usando resultado salvo (0 GPU gasto).")
            return r

    result = run_experiment(run_name=run_name, **kwargs)
    cache.append(result)
    save_cached_results(cache)
    print(f"[cache] '{run_name}' salvo em {RESULTS_PATH}.")
    return result


all_results = load_cached_results()

## 3. Semear o cache com os resultados que você já obteve

Os Experimentos 1, 2 e 3 (congelamento) **já foram treinados por você** na
rodada anterior, para os dois modelos, com a base de treino completa. Para
não desperdiçar esse trabalho (nem GPU), a célula abaixo grava esses
resultados diretamente no cache — só treina de fato o que ainda não existir.

Se o cache (`RESULTS_PATH`) já tiver esses resultados (por exemplo, porque
você está usando o Google Drive e já rodou isso antes), a célula não
duplica nada.

In [ ]:
JA_OBTIDOS = [
]

# ATENÇÃO: os val_accuracy/val_f1 acima ficaram como None porque não foram
# capturados na rodada anterior. Se você ainda tiver esses números (ex: no
# log de saída da célula antiga), pode preenchê-los manualmente — eles só
# são usados para a escolha da melhor config no Experimento 4, não para os
# resultados finais de teste exigidos pelo enunciado.

existing_names = {r["run_name"] for r in all_results}
added = 0
for r in JA_OBTIDOS:
    if r["run_name"] not in existing_names:
        all_results.append(r)
        added += 1

if added:
    save_cached_results(all_results)
print(f"{added} resultado(s) semeado(s) no cache. Total no cache agora: {len(all_results)}.")

## 4. Função de métricas (`compute_metrics`)

Accuracy, Precision, Recall e F1 via `scikit-learn`, para a classe positiva
(label = 1), como exige o enunciado.

In [ ]:
def compute_metrics(eval_pred):
    """Accuracy, Precision, Recall e F1 para a classe positiva (label=1)."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## 5. Utilitários de congelamento de camadas

Iguais à versão anterior — funcionam para os dois modelos (ambos são
`BertForSequenceClassification`).

In [ ]:
def freeze_all_except_classifier(model):
    """Experimento 1: congela todo o transformer, treina só a cabeça."""
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith("classifier")
    return model


def unfreeze_last_n_layers(model, n_layers=3):
    """Experimento 2: libera apenas as últimas `n_layers` do encoder."""
    layer_indices = {
        int(re.search(r"encoder\.layer\.(\d+)\.", name).group(1))
        for name, _ in model.named_parameters()
        if re.search(r"encoder\.layer\.(\d+)\.", name)
    }
    total_layers = max(layer_indices) + 1
    unfreeze_from = total_layers - n_layers

    for name, param in model.named_parameters():
        if name.startswith("classifier"):
            param.requires_grad = True
            continue
        match = re.search(r"encoder\.layer\.(\d+)\.", name)
        if match and int(match.group(1)) >= unfreeze_from:
            param.requires_grad = True
        else:
            param.requires_grad = False

    print(
        f"[Exp. 2] {n_layers} de {total_layers} camadas do encoder descongeladas "
        f"(camadas {unfreeze_from} a {total_layers - 1})."
    )
    return model, total_layers


def unfreeze_all(model):
    """Experimento 3: fine-tuning completo (todas as camadas treináveis)."""
    for _, param in model.named_parameters():
        param.requires_grad = True
    return model


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

## 6. Rotina genérica de experimento (`run_experiment`)

Igual à anterior, com um campo novo no resultado: `n_train_examples`
(quantas amostras de treino foram realmente usadas — relevante agora que o
Experimento 4 usa uma subamostra).

In [ ]:
def run_experiment(
    model_name,
    train_dataset,
    val_dataset,
    test_dataset,
    run_name,
    freeze_strategy="full",   # "head_only" | "partial" | "full"
    n_unfrozen_layers=3,
    num_epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    gradient_accumulation_steps=1,
    max_train_samples=None,   # subamostra do treino; None = usa tudo
    output_dir="./results",
):
    if max_train_samples is not None:
        train_dataset = train_dataset.shuffle(seed=42).select(
            range(min(max_train_samples, len(train_dataset)))
        )

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    n_layers_total = None
    if freeze_strategy == "head_only":
        freeze_all_except_classifier(model)
    elif freeze_strategy == "partial":
        model, n_layers_total = unfreeze_last_n_layers(model, n_unfrozen_layers)
    elif freeze_strategy == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"freeze_strategy inválido: {freeze_strategy}")

    trainable_params, total_params = count_trainable_params(model)

    training_args = TrainingArguments(
        output_dir=f"{output_dir}/{run_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        weight_decay=0.01,
        save_strategy="no",
        logging_steps=200,
        fp16=torch.cuda.is_available(),
        report_to="none",
        **{EVAL_STRATEGY_KEY: "epoch"},
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    print(f"\n=== Treinando: {run_name} | {trainable_params:,}/{total_params:,} parâmetros "
          f"treináveis | {len(train_dataset):,} exemplos de treino ===")
    start_time = time.time()
    trainer.train()
    train_time_seconds = time.time() - start_time

    val_metrics = trainer.evaluate(eval_dataset=val_dataset)
    test_metrics = trainer.evaluate(eval_dataset=test_dataset)

    result = {
        "run_name": run_name,
        "model_name": model_name,
        "freeze_strategy": freeze_strategy,
        "n_unfrozen_layers": n_unfrozen_layers if freeze_strategy == "partial" else None,
        "total_layers": n_layers_total,
        "num_epochs": num_epochs,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "pct_trainable": round(100 * trainable_params / total_params, 2),
        "n_train_examples": len(train_dataset),
        "train_time_seconds": round(train_time_seconds, 1),
        "train_time_minutes": round(train_time_seconds / 60, 2),
        "val_accuracy": val_metrics["eval_accuracy"],
        "val_f1": val_metrics["eval_f1"],
        "test_accuracy": test_metrics["eval_accuracy"],
        "test_precision": test_metrics["eval_precision"],
        "test_recall": test_metrics["eval_recall"],
        "test_f1": test_metrics["eval_f1"],
    }

    del trainer, model
    clear_gpu_memory()

    return result

## 7. Configuração dos modelos e datasets

In [ ]:
MODELS = {
    "Modelo 1": dict(
        model_name=model_name_1,
        train=tokenized_train_1,
        val=tokenized_val_1,
        test=tokenized_test_1,
    ),
    "Modelo 2": dict(
        model_name=model_name_2,
        train=tokenized_train_2,
        val=tokenized_val_2,
        test=tokenized_test_2,
    ),
}

for label, cfg in MODELS.items():
    print(f"{label}: {cfg['model_name']}  "
          f"(treino={len(cfg['train'])}, val={len(cfg['val'])}, teste={len(cfg['test'])})")

## 8. Experimentos 1, 2 e 3 — Estratégias de Congelamento

Com o cache já semeado na seção 3, esta célula **não deve gastar GPU** — ela
só vai imprimir `[cache] ... já treinado` para os 6 experimentos. Ela só
treina de verdade se algum desses resultados faltar no cache (por exemplo,
se você limpar o CSV ou quiser adicionar um terceiro modelo no futuro).

In [ ]:
FREEZE_EXPERIMENTS = {
    "exp1_head_only": dict(freeze_strategy="head_only"),
    "exp2_partial_3layers": dict(freeze_strategy="partial", n_unfrozen_layers=3),
    "exp3_full_finetuning": dict(freeze_strategy="full"),
}

for model_label, data in MODELS.items():
    for exp_name, cfg in FREEZE_EXPERIMENTS.items():
        res = run_or_load(
            all_results,
            run_name=f"{model_label}_{exp_name}_base",
            model_name=data["model_name"],
            train_dataset=data["train"],
            val_dataset=data["val"],
            test_dataset=data["test"],
            num_epochs=3,
            batch_size=16,
            **cfg,
        )

# Create DataFrame and add 'model_label' and 'experiment' columns
results_df = pd.DataFrame(all_results)

# Dynamically extract 'model_label' based on the keys in MODELS
if "model_label" not in results_df.columns:
    def extract_model_label_from_run_name(run_name):
        for label in MODELS.keys():
            if run_name.startswith(f"{label}_"):
                return label
        return None
    results_df["model_label"] = results_df["run_name"].apply(extract_model_label_from_run_name)

# Dynamically extract 'experiment' by removing model_label and '_base'
if "experiment" not in results_df.columns:
    def extract_experiment_from_run_name(row):
        model_label = row["model_label"]
        run_name = row["run_name"]
        if model_label and run_name.startswith(f"{model_label}_"):
            return run_name.replace(f"{model_label}_", "").replace("_base", "")
        return None
    results_df["experiment"] = results_df.apply(extract_experiment_from_run_name, axis=1)


results_to_display = results_df[
    ["model_label", "experiment", "trainable_params", "pct_trainable",
     "n_train_examples", "train_time_minutes",
     "test_accuracy", "test_precision", "test_recall", "test_f1"]
]
results_to_display

## 9. Experimento 4 — Número de Épocas (versão leve)

Para caber na cota gratuita do Colab, a busca de épocas usa uma
**subamostra do treino** (`TRAIN_SUBSAMPLE_SIZE`), mantendo os conjuntos de
validação e teste **completos** (a avaliação é muito mais barata que o
treino, então não precisa ser reduzida). Isso é uma decisão de engenharia
por restrição de recurso computacional — documente isso no formulário de
resultados.

- `TRAIN_SUBSAMPLE_SIZE = 12000` (~16% dos dados de treino) é um bom ponto
  de partida: ainda é uma quantidade grande de exemplos para fine-tuning de
  um BERT, e reduz o tempo de treino em ~6x em relação ao dataset completo.
- Se sua cota permitir, aumente esse valor (ex: `20000` ou `30000`) para
  resultados mais próximos dos Experimentos 1–3 (que usaram o dataset
  inteiro).
- Se ainda estourar a cota, diminua para `6000`–`8000`.

Assim como no passo anterior, cada combinação (modelo × nº de épocas) só é
treinada uma vez — reexecuções pulam o que já está no cache.

In [ ]:
TRAIN_SUBSAMPLE_SIZE = 30000  # ajuste conforme sua cota de GPU disponível
EPOCHS_GRID = [2, 3, 5]

for model_label, data in MODELS.items():
    for n_epochs in EPOCHS_GRID:
        res = run_or_load(
            all_results,
            run_name=f"{model_label}_exp4_epochs{n_epochs}_sub{TRAIN_SUBSAMPLE_SIZE}",
            model_name=data["model_name"],
            train_dataset=data["train"],
            val_dataset=data["val"],
            test_dataset=data["test"],
            freeze_strategy="full",
            num_epochs=n_epochs,
            batch_size=16,
            max_train_samples=TRAIN_SUBSAMPLE_SIZE,
        )

epoch_df = pd.DataFrame(
    [r for r in all_results if r["run_name"].startswith(tuple(
        f"{m}_exp4_epochs" for m in MODELS
    ))]
)

# Adiciona a coluna 'model_label' ao epoch_df extraindo do 'run_name'
if "model_label" not in epoch_df.columns:
    epoch_df["model_label"] = epoch_df["run_name"].apply(lambda x: x.split('_')[0])

best_epochs = (
    epoch_df.sort_values("val_f1", ascending=False)
    .groupby("model_label", as_index=False)
    .first()[["model_label", "num_epochs", "n_train_examples", "val_f1", "test_f1", "train_time_minutes"]]
)

print("Melhor número de épocas por modelo (escolhido pelo F1 de VALIDAÇÃO, na subamostra):")
best_epochs

### [Opcional] Refinar a melhor configuração do Exp. 4 no dataset completo

Se sua cota de GPU permitir, um passo extra recomendado (mas não
obrigatório pelo enunciado) é treinar **apenas a melhor configuração**
encontrada acima com o dataset de treino completo, para ter um número final
mais robusto. Isso custa só 1 treinamento por modelo, não 3.

In [ ]:
RUN_FULL_DATA_REFINEMENT = False  # mude para True se quiser (e tiver cota)

if RUN_FULL_DATA_REFINEMENT:
    for _, row in best_epochs.iterrows():
        model_label = row["model_label"]
        data = MODELS[model_label]
        n_epochs = int(row["num_epochs"])
        res = run_or_load(
            all_results,
            run_name=f"{model_label}_exp4_best_epochs{n_epochs}_fulldata",
            model_name=data["model_name"],
            train_dataset=data["train"],
            val_dataset=data["val"],
            test_dataset=data["test"],
            freeze_strategy="full",
            num_epochs=n_epochs,
            batch_size=16,
        )
else:
    print("Refinamento com dataset completo desativado (RUN_FULL_DATA_REFINEMENT = False).")

## 10. Experimento 5 — Comparação entre Modelos

Consolida tudo o que está no cache (Experimentos 1–4, os dois modelos) em
uma única tabela.

In [ ]:
results_df = pd.DataFrame(all_results)

comparison_cols = [
    "model_label", "model_name", "experiment", "freeze_strategy",
    "n_unfrozen_layers", "num_epochs", "trainable_params", "pct_trainable",
    "n_train_examples", "train_time_minutes",
    "test_accuracy", "test_precision", "test_recall", "test_f1",
]
# nem toda linha tem "experiment"/"model_label" caso venha só do CSV bruto;
# preenche a partir do run_name como segurança
if "experiment" not in results_df.columns:
    results_df["experiment"] = results_df["run_name"].str.split("_", n=1).str[1]
if "model_label" not in results_df.columns:
    results_df["model_label"] = results_df["run_name"].str.split("_", n=1).str[0]

results_table = results_df[comparison_cols].sort_values(["model_label", "experiment"]).reset_index(drop=True)
results_table

pivot_f1 = results_df.pivot_table(index="experiment", columns="model_label", values="test_f1")
pivot_acc = results_df.pivot_table(index="experiment", columns="model_label", values="test_accuracy")
pivot_time = results_df.pivot_table(index="experiment", columns="model_label", values="train_time_minutes")

print("F1 (teste) por experimento:")
display(pivot_f1)
print("\nAccuracy (teste) por experimento:")
display(pivot_acc)
print("\nTempo de treinamento (min) por experimento:")
display(pivot_time)

### [Opcional] Gráfico comparativo para a apresentação

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pivot_f1.plot(kind="bar", ax=ax)
ax.set_ylabel("F1-score (conjunto de teste)")
ax.set_xlabel("Experimento")
ax.set_title("Comparação de F1 por experimento e por modelo")
ax.legend(title="Modelo")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Exportar resultados (para o formulário do trabalho)

In [ ]:
results_df.to_csv("resultados_experimentos_final.csv", index=False)
print("Resultados exportados para resultados_experimentos_final.csv "
      f"(o cache completo também está em {RESULTS_PATH}).")
results_df.head()